# **Поиск вакансий по резюме. Сравнение**

In [ ]:
# Установка библиотеки (если ещё не установлена)
#!pip install python-docx

In [ ]:
#pip install --upgrade transformers

In [1]:
PINECONE_API_KEY = "pcsk_2ghPWB_3v4p9rrdZ85hp2SzJnMMEPELbpE1cPiXhXUaiBQueTq7C3Wy1ag271dGJnbbdG7" # акк Анны

vectorizing_model = 'all-MiniLM-L6-v2'
#vectorizing_model = 'sentence-transformers/distilbert-base-nli-mean-tokens'

index_name = 'res-vac-match-2-scill'
index_dimension=384

## Извлечение навыков по словарю патетернов

# **Часть первая - Наполнение векторной базы**

# Извлечение вакансий из таблицы

In [2]:
from scills import extract_skills, all_patterns

✅ Паттерны загружены


In [3]:
# Проверка на одной вакансии
vacancy_text_test = """
Primary Purpose: Development and maintenance of software applications built using Microsoft stack.
 Essential Duties and Responsibilities: Following is a summary of the essential functions for this job.
 Other duties may be performed, both major and minor, which are not mentioned below.
 Specific activities may change from time to time.
 Design, develop and test software applications Maintain systems by identifying and correcting software defects Create technical specifications and unit test plans Work with QA to align understanding of requirements and to develop system test plans Work as part of an Agile development team to solve problems and develop projects in a fast paced environment 
 Follow instructions and pre-established guidelines to perform the functions of the job 
 Demonstrate a basic degree of creativity and problem solving skills 
 Follow the established software development life cycle Follow established coding standards and naming conventions Support applications using software development methodologies including structured programming, documentation, design and code review 
 Work with business analysts and application users to define and design robust user centric application solutions 
 Collaborate with UX resources to drive consistent look and feel of user interface 
 Collaborate with database resources to ensure robust and complete data access and manipulation 
 Collaborate with IT Operations to ensure hardware and software are aligned to deliver business requirements 
 Mentor junior and other new developers On-site regular attendance and punctuality are essential functions of the job 
 Minimum Skills and Competencies: The requirements listed below are representative of the knowledge, skill and/or ability required. 
 Reasonable accommodations may be made to enable individuals with disabilities to perform the essential functions. 
 Bachelor's Degree or in-lieu of degree equivalent education, training and work-related experience High school diploma or general education degree (GED) required 5+ years of experience in Microsoft technology stack 
 Expert in C#, JavaScript, MSSQL 2012 or above Proficient with MVC, Angular, Asp.Net, JQuery 
 Hands on experience with Visual Studio & TFS as source control tool 
 Must have good understanding of SOLID object oriented programming (OOP) concepts 
 Must have experience with WCF Experience in end-to-end Software Development 
 Life Cycle (SDLC) project Experience working with 
 Agile/Scrum methodologies Good at understanding requirements, getting clarifications 
 Have passion for learning new technologies and enhancing existing skills 
 Proactive issue resolution with a positive attitude Understand solution at the project level 
 Effective organization and time management skills with the ability to work under pressure and adhere to project deadlines 
 Excellent interpersonal skills with the ability to establish working relationships with individuals at varying levels within the organization 
 Demonstrated integrity within a professional environment 
 Ability to adapt to new situations and learn quickly 
 Must perform well in high-energy, dynamic and team-oriented environments 
 Must possess effective verbal and written communication skills 
 Desired Skills: Insurance industry experience 
 Familiarity with writing unit tests Relevant Microsoft certification 
 Experience with Web API, Entity Framework Experience with Crystal Reports, SSRS and/or SSIS 
 - About National General Holdings: 
 National General Holdings Corp. is headquartered in New York City. 
 ational General traces its roots to 1939, has a financial strength rating of A- (excellent) from A.M. 
 Best, and provides personal and commercial automobile, homeowners, umbrella, recreational vehicle, motorcycle, supplemental health, and other niche insurance products. 
 We are a specialty personal lines insurance holding company. 
 Through our subsidiaries, we provide a variety of insurance products, including personal and commercial automobile, homeowners, umbrella, recreational vehicle, supplemental health, lender-placed and other niche insurance products. 
 National General is a fast paced, dynamic, and entrepreneurial organization. 
 Our team members live by and exemplify what we call the 4Es. 
 At National General, we are Energized, Engaged, Empowered, and we Execute every day in order to provide an exceptional experience for our customers! 
 We are passionate about our organization and the value that we add every day. 
 A successful candidate with National General will embody the 4Es. Our team is poised to outperform the competition. 
 We are National General Insurance and with us you can be extraordinary! 
 Come join our team! Companies and Partners Direct General Auto & Life, Personal Express Insurance, Century-National Insurance, ABC Insurance Agencies, NatGen Preferred, NatGen Premier, Seattle Specialty, National General Lender Services, ARS, RAC Insurance Partners, Mountain Valley Indemnity, New Jersey Skylands, Adirondack Insurance Exchange, VelaPoint, Quotit, HealthCompare, AHCP, NHIC, Healthcare Solutions Team, North Star Marketing, Euro Accident. 
 In addition to a phenomenal career opportunity, National General Insurance offers an excellent benefits package including: * Paid Training * Medical, Dental, Vision benefits * Wellness Programs * Life and Short/Long Term Disability Insurance * 401k w/ Company Match * Company Paid Holidays & Generous Time-off policy * Employee Discount Program * Career Advancement and Development Opportunities * On-site Healthcare Clinic (Winston-Salem and Cleveland offices) * On-site Fitness Center (Cleveland and Dallas offices) * Subsidized parking (Cleveland office) National General Holdings Corp. is an Equal Opportunity (EO) employer 
 - Veterans/Disabled and other protected categories. 
 All qualified applicants will receive consideration for employment regardless of any characteristic protected by law. 
 Candidates must possess authorization to work in the United States, as it is not our practice to sponsor individuals for work visas. 
 In the event you need assistance or accommodation in completing your online application, please contact NGIC main office by phone at (336) 435-2000. -"
"""
print(extract_skills(vacancy_text_test, all_patterns))

agile, angular, dotnet, entity_framework, express, javascript, jquery, mssql, mvc, oop, scrum, sdlc, solid, tfs, wcf, webapi


In [4]:
import pandas as pd
import os

# Путь к папке с датасетом
DATASET_PATH = "vacancy-resume-matching-dataset-main"
csv_file = os.path.join(DATASET_PATH, "5_vacancies.csv")

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"Файл не найден: {csv_file}\nУбедитесь, что папка 'vacancy-resume-matching-dataset-main' находится в текущей директории.")

# Загружаем CSV
df_vacancies = pd.read_csv(csv_file)

# Извлекаем нужные поля и удаляем пустые строки
df_vacancies = df_vacancies[['job_title', 'job_description']].dropna()

# Создаём список chunk_text: каждый элемент — один текст вакансии
chunk_text = []
metadata = []

for idx, row in df_vacancies.iterrows():
    title = row['job_title']
    desc = row['job_description']
    
    # Формируем единый текст для вакансии (один чанк = одна вакансия)
    full_text = f"Job title: {title}. Job description: {desc}"
    # Извлекаем навыки
    full_text = extract_skills(full_text, all_patterns)
    chunk_text.append(full_text)
    metadata.append({
        "job_title": title,
        "job_description": desc,
        "vacancy_id": idx+1
    })

# Выводим статистику
print(f"Создано {len(chunk_text)} чанков (по одному на вакансию) из {len(df_vacancies)} вакансий.")

# Пример первого чанка
print("\nПример первого чанка:")
print(chunk_text[0])
print("Метаданные:")
print(metadata[0])


Создано 5 чанков (по одному на вакансию) из 5 вакансий.

Пример первого чанка:
agile, angular, dotnet, entity_framework, express, javascript, jquery, mssql, mvc, oop, scrum, sdlc, solid, tfs, wcf, webapi
Метаданные:
{'job_title': 'Software Developer - .Net', 'job_description': "Primary Purpose: Development and maintenance of software applications built using Microsoft stack. Essential Duties and Responsibilities: Following is a summary of the essential functions for this job. Other duties may be performed, both major and minor, which are not mentioned below. Specific activities may change from time to time. Design, develop and test software applications Maintain systems by identifying and correcting software defects Create technical specifications and unit test plans Work with QA to align understanding of requirements and to develop system test plans Work as part of an Agile development team to solve problems and develop projects in a fast paced environment Follow instructions and pre-

## Подготовка к загрузке в Pinecone

In [5]:

from sentence_transformers import SentenceTransformer

# Загружаем модель
model = SentenceTransformer(vectorizing_model)

# Генерируем эмбеддинги
embeddings = model.encode(chunk_text).tolist()  # Список списков, длина 768

# Генерируем уникальные ID для векторов
ids = [f"chunk_{i}" for i in range(len(chunk_text))]

# Подготавливаем записи для upsert
my_chunks = list(zip(ids, embeddings, metadata))

print(f"✅ Готово к загрузке: {len(my_chunks)} векторов в Pinecone")


✅ Готово к загрузке: 5 векторов в Pinecone


## Загрузка 5-и чанков в Pinecone

In [6]:
from pinecone import Pinecone, ServerlessSpec

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)


# Создание индекса, если его нет
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=index_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Подключение к индексу
index = pc.Index(index_name)

# Загружаем все чанки из my_chunks (всего 5 — можно без батчей)
print(f"📤 Загружаем {len(my_chunks)} векторов в Pinecone...")
upsert_response = index.upsert(vectors=my_chunks)

# Проверка результата
print(f"✅ Успешно загружено {upsert_response['upserted_count']} векторов.")


📤 Загружаем 5 векторов в Pinecone...
✅ Успешно загружено 5 векторов.


# **Часть вторая - Поиск вакансии по резюме**

### Извлечение текста из вордовских файлов

In [7]:
from docx import Document
import os

def extract_text_from_docx(folder_path, file_number):
    """
    Извлекает текст из файла {file_number}.docx в указанной папке.
    Обрабатывает как параграфы, так и текст внутри таблиц.

    Args:
        folder_path (str): Путь к папке с файлами (например, r"C:\CV")
        file_number (int): Номер файла (1 до 65)

    Returns:
        str: Извлечённый текст из .docx файла или пустая строка, если файл не найден/ошибка
    """
    file_name = f"{file_number}.docx"
    file_path = os.path.join(folder_path, file_name)

    try:
        doc = Document(file_path)
        full_text = []

        # 1. Извлекаем текст из параграфов
        for paragraph in doc.paragraphs:
            if paragraph.text.strip():  # Только непустые параграфы
                full_text.append(paragraph.text.strip())

        # 2. Извлекаем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    for paragraph in cell.paragraphs:
                        if paragraph.text.strip():  # Только непустые параграфы в ячейках
                            full_text.append(paragraph.text.strip())


        # Объединяем все тексты с разрывами строк
        
        return '\n'.join(full_text)

    except FileNotFoundError:
        print(f"⚠️ Файл не найден: {file_path}")
        return ""
    except Exception as e:
        print(f"❌ Ошибка при чтении {file_path}: {e}")
        return ""


### Генерация поискового списка и поиск

In [9]:
import os
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone

# Инициализация модели для векторизации
model = SentenceTransformer(vectorizing_model)

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Путь к папке CV
CV_FOLDER_PATH = r"vacancy-resume-matching-dataset-main\CV"

# Проверим, существует ли папка
if not os.path.exists(CV_FOLDER_PATH):
    print(f"❌ Папка не существует: {CV_FOLDER_PATH}")
    print("Убедитесь, что путь указан правильно. Возможно, нужно использовать абсолютный путь.")
else:
    print(f"✅ Папка найдена: {CV_FOLDER_PATH}\n")

    # Собираем тексты всех файлов и ищем топ-5 вакансий для каждого
    search_results = []
    
    for i in range(1, 66):  # от 1 до 65 включительно
        print(f"📄 Обработка файла {i}.docx...")
        
        # Извлекаем текст резюме
        text = extract_text_from_docx(CV_FOLDER_PATH, i)
        
        # 3. Извлекаем навыки
#        text = extract_skills(text, all_patterns)
        # Проверить, что текст файла не пустой

        print(text[:70], "...")
        if text:
            # Векторизуем резюме
            resume_embedding = model.encode([text]).tolist()[0]
            
            # Ищем топ-5 вакансий в Pinecone
            search_response = index.query(
                vector=resume_embedding,
                top_k=5,
                include_metadata=True
            )
            
            # Извлекаем порядковые номера вакансий из metadata (vacancy_id)
            # Вакансии уже отсортированы по релевантности (от самой релевантной к наименее)
            top_5_vacancies = []
            for match in search_response.matches:
                vacancy_id = int(match.metadata["vacancy_id"])  # Преобразуем в целое число
                top_5_vacancies.append(vacancy_id)
            
            search_results.append(top_5_vacancies)
            print(f"   ✅ Найдено вакансий: {top_5_vacancies}")
        else:
            # Если текст не извлекся, добавляем пустой список
            search_results.append([])
            print(f"   ⚠️ Текст не извлечен")

    print(f"\n=== Готово! ===")
    print(f"Обработано резюме: {len([r for r in search_results if r])} из 65")
    


✅ Папка найдена: vacancy-resume-matching-dataset-main\CV

📄 Обработка файла 1.docx...
****************
Java full stack developer
(60% – IN JAVA BACKEND DEVE ...
   ✅ Найдено вакансий: [1, 4, 2, 5, 3]
📄 Обработка файла 2.docx...
*************
FULL-STACK DEVELOPER
4 years’ experience (80 % in JAVA B ...
   ✅ Найдено вакансий: [4, 2, 1, 5, 3]
📄 Обработка файла 3.docx...
*************************
5 years of total experience in development w ...
   ✅ Найдено вакансий: [1, 2, 4, 5, 3]
📄 Обработка файла 4.docx...
**************
JAVA BACKEND DEVELOPER WITH 3 YEARS OF HANDS-ON EXPERIE ...
   ✅ Найдено вакансий: [2, 4, 1, 5, 3]
📄 Обработка файла 5.docx...
6 years of proven experience
(4 years in Java Backend/1 year in NodeJS ...
   ✅ Найдено вакансий: [1, 4, 2, 5, 3]
📄 Обработка файла 6.docx...
LEONID SHALAEV FULL STACK DEVELOPER with 1+ year experience
70% Back-e ...
   ✅ Найдено вакансий: [4, 2, 1, 5, 3]
📄 Обработка файла 7.docx...
***************
FRONTEND DEVELOPER
2 years of experience in fr

In [12]:
# Выводим структуру результатов (аналогично ANNOTATOR_1_RANKINGS)
print(f"\n📊 Результаты поиска по всем словарям паттернов, навыки извлекаются только из вакансий (модель - {vectorizing_model}, размер - {index_dimension}):")
print("SEARCH_RESULTS = [\n", end="")
for i in range(0, len(search_results), 5):
    group = search_results[i:i+5]
    
    # Формируем комментарий
    start_num = i + 1
    end_num = min(i + 5, len(search_results))
    comment = f"#{start_num}-{end_num}" if end_num - start_num + 1 == 5 else f"#{start_num}-{end_num}"
    
    # Выводим первую строку группы с комментарием
    if i == 0:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
    else:
        print(f"{group[0]},{group[1]},{group[2]},{group[3]},{group[4]}, {comment}")
print("]")



📊 Результаты поиска по всем словарям паттернов, навыки извлекаются только из вакансий (модель - all-MiniLM-L6-v2, размер - 384):
SEARCH_RESULTS = [
[1, 4, 2, 5, 3],[4, 2, 1, 5, 3],[1, 2, 4, 5, 3],[2, 4, 1, 5, 3],[1, 4, 2, 5, 3], #1-5
[4, 2, 1, 5, 3],[2, 1, 4, 5, 3],[2, 1, 4, 5, 3],[1, 2, 4, 3, 5],[2, 1, 4, 3, 5], #6-10
[3, 2, 5, 1, 4],[4, 1, 2, 5, 3],[2, 1, 4, 3, 5],[4, 1, 2, 5, 3],[1, 4, 2, 5, 3], #11-15
[4, 1, 2, 5, 3],[4, 1, 2, 5, 3],[1, 4, 2, 5, 3],[1, 2, 4, 5, 3],[4, 2, 1, 5, 3], #16-20
[4, 1, 2, 5, 3],[5, 3, 2, 4, 1],[2, 4, 3, 5, 1],[4, 1, 2, 5, 3],[1, 4, 2, 5, 3], #21-25
[4, 3, 2, 5, 1],[1, 2, 4, 3, 5],[2, 1, 4, 5, 3],[1, 4, 2, 5, 3],[2, 4, 1, 5, 3], #26-30
[4, 1, 2, 5, 3],[3, 4, 1, 2, 5],[4, 3, 2, 1, 5],[4, 2, 1, 5, 3],[1, 2, 4, 5, 3], #31-35
[3, 2, 4, 5, 1],[1, 2, 4, 5, 3],[4, 5, 1, 2, 3],[3, 4, 2, 5, 1],[1, 2, 4, 5, 3], #36-40
[1, 4, 2, 5, 3],[4, 2, 1, 3, 5],[4, 2, 5, 1, 3],[2, 4, 1, 5, 3],[4, 2, 1, 3, 5], #41-45
[1, 4, 2, 5, 3],[1, 4, 5, 3, 2],[1, 4, 5, 3, 2],[2, 4, 1, 5, 3

### 🔥 Удаление индекса 

In [ ]:
'''
# Инициализация модели для векторизации
model = SentenceTransformer(vectorizing_model)

# Инициализация Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
#index = pc.Index(index_name)
'''

In [11]:
# Удалить индекс через клиент
pc.delete_index(index_name)

print(f"✅ Индекс '{index_name}' успешно удалён.")


✅ Индекс 'res-vac-match-2-scill' успешно удалён.
